GSAC with UpgradedEnvHRL and GPUReplayBufferHER

In [ ]:
import os
import random
import time
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

from UpgradedEnvHRL import UpgradedEnvHRL
from gpu_replay_buffer_her import GPUReplayBufferHER

# Optional: suppress noisy warnings in notebook output
os.environ.setdefault("PYTHONWARNINGS", "ignore")


In [ ]:
@dataclass
class Config:
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Env
    horizon: int = 150
    control_freq: int = 20

    # Replay / training
    replay_size: int = 300_000
    warmup_steps: int = 5_000
    total_steps: int = 120_000
    batch_size: int = 256
    updates_per_step: int = 1

    # SAC
    gamma: float = 0.99
    tau: float = 0.005
    lr: float = 3e-4
    hidden_dim: int = 256

    # Entropy
    init_alpha: float = 0.2

    # HER
    her_ratio: float = 0.8
    her_success_reward: float = 100.0

    # Logging
    log_every: int = 2_000
    eval_every: int = 20_000
    eval_episodes: int = 5

CFG = Config()
print(CFG)


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
device = torch.device(CFG.device)
print("device:", device)


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class GaussianActor(nn.Module):
    LOG_STD_MIN = -20
    LOG_STD_MAX = 2

    def __init__(self, obs_dim, act_dim, hidden_dim):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden_dim, act_dim)
        self.log_std = nn.Linear(hidden_dim, act_dim)

    def forward(self, obs, deterministic=False, with_logprob=True):
        h = self.backbone(obs)
        mu = self.mu(h)
        log_std = torch.clamp(self.log_std(h), self.LOG_STD_MIN, self.LOG_STD_MAX)
        std = torch.exp(log_std)

        dist = Normal(mu, std)
        pre_tanh = mu if deterministic else dist.rsample()

        logp = None
        if with_logprob:
            logp = dist.log_prob(pre_tanh).sum(dim=-1, keepdim=True)
            logp -= (2 * (np.log(2) - pre_tanh - F.softplus(-2 * pre_tanh))).sum(dim=-1, keepdim=True)

        action = torch.tanh(pre_tanh)
        return action, logp


class DoubleCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim):
        super().__init__()
        self.q1 = MLP(obs_dim + act_dim, hidden_dim, 1)
        self.q2 = MLP(obs_dim + act_dim, hidden_dim, 1)

    def forward(self, obs, act):
        x = torch.cat([obs, act], dim=-1)
        return self.q1(x), self.q2(x)


In [ ]:
class GSACAgent:
    def __init__(self, obs_dim, act_dim, cfg: Config):
        self.cfg = cfg
        self.device = torch.device(cfg.device)
        self.act_dim = act_dim

        self.actor = GaussianActor(obs_dim, act_dim, cfg.hidden_dim).to(self.device)
        self.critic = DoubleCritic(obs_dim, act_dim, cfg.hidden_dim).to(self.device)
        self.critic_targ = DoubleCritic(obs_dim, act_dim, cfg.hidden_dim).to(self.device)
        self.critic_targ.load_state_dict(self.critic.state_dict())

        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=cfg.lr)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=cfg.lr)

        self.log_alpha = torch.tensor(np.log(cfg.init_alpha), device=self.device, requires_grad=True)
        self.alpha_opt = torch.optim.Adam([self.log_alpha], lr=cfg.lr)
        self.target_entropy = -float(act_dim)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    @torch.no_grad()
    def act(self, obs_np, deterministic=False):
        obs = torch.as_tensor(obs_np, dtype=torch.float32, device=self.device).unsqueeze(0)
        action, _ = self.actor(obs, deterministic=deterministic, with_logprob=False)
        return action.squeeze(0).cpu().numpy()

    def soft_update(self):
        tau = self.cfg.tau
        with torch.no_grad():
            for p, p_t in zip(self.critic.parameters(), self.critic_targ.parameters()):
                p_t.data.mul_(1 - tau).add_(tau * p.data)

    def update(self, replay: GPUReplayBufferHER, batch_size: int):
        b = replay.sample(batch_size)
        obs = b["obs"]
        act = b["actions"]
        rew = b["rewards"]
        next_obs = b["next_obs"]
        done = b["dones"]

        # Critic update
        with torch.no_grad():
            next_act, next_logp = self.actor(next_obs, deterministic=False, with_logprob=True)
            q1_t, q2_t = self.critic_targ(next_obs, next_act)
            q_t = torch.min(q1_t, q2_t) - self.alpha.detach() * next_logp
            y = rew + self.cfg.gamma * (1.0 - done) * q_t

        q1, q2 = self.critic(obs, act)
        critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)

        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()
        self.critic_opt.step()

        # Actor update
        new_act, logp = self.actor(obs, deterministic=False, with_logprob=True)
        q1_pi, q2_pi = self.critic(obs, new_act)
        q_pi = torch.min(q1_pi, q2_pi)
        actor_loss = (self.alpha.detach() * logp - q_pi).mean()

        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        self.actor_opt.step()

        # Alpha update
        alpha_loss = -(self.log_alpha * (logp + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad(set_to_none=True)
        alpha_loss.backward()
        self.alpha_opt.step()

        self.soft_update()

        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": float(actor_loss.item()),
            "alpha": float(self.alpha.item()),
            "alpha_loss": float(alpha_loss.item()),
        }

    def save(self, path):
        torch.save({
            "actor": self.actor.state_dict(),
            "critic": self.critic.state_dict(),
            "critic_targ": self.critic_targ.state_dict(),
            "actor_opt": self.actor_opt.state_dict(),
            "critic_opt": self.critic_opt.state_dict(),
            "log_alpha": self.log_alpha.detach().cpu(),
            "alpha_opt": self.alpha_opt.state_dict(),
        }, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.actor.load_state_dict(ckpt["actor"])
        self.critic.load_state_dict(ckpt["critic"])
        self.critic_targ.load_state_dict(ckpt["critic_targ"])
        self.actor_opt.load_state_dict(ckpt["actor_opt"])
        self.critic_opt.load_state_dict(ckpt["critic_opt"])
        self.log_alpha.data.copy_(ckpt["log_alpha"].to(self.device))
        self.alpha_opt.load_state_dict(ckpt["alpha_opt"])


In [ ]:
# Build training env
env = UpgradedEnvHRL(
    render=False,
    domain_randomization=True,
    control_freq=CFG.control_freq,
    horizon=CFG.horizon,
    controller="OSC_POSE",
    robot_name="Rover2026",
)

obs, info = env.reset(seed=CFG.seed)
obs_dim = int(obs.shape[0])
act_dim = int(env.action_space.shape[0])

agent = GSACAgent(obs_dim=obs_dim, act_dim=act_dim, cfg=CFG)
replay = GPUReplayBufferHER(
    capacity=CFG.replay_size,
    obs_dim=obs_dim,
    action_dim=act_dim,
    device=CFG.device,
    her_ratio=CFG.her_ratio,
    her_success_reward=CFG.her_success_reward,
)

print("obs_dim:", obs_dim, "act_dim:", act_dim)


In [ ]:
def run_eval(agent: GSACAgent, n_episodes=5):
    eval_env = UpgradedEnvHRL(
        render=False,
        domain_randomization=False,
        control_freq=CFG.control_freq,
        horizon=CFG.horizon,
        controller="OSC_POSE",
        robot_name="Rover2026",
    )
    returns = []
    successes = []
    for ep in range(n_episodes):
        obs, info = eval_env.reset(seed=CFG.seed + 1000 + ep)
        done = False
        ep_ret = 0.0
        while not done:
            a = agent.act(obs, deterministic=True)
            obs, r, terminated, truncated, inf = eval_env.step(a)
            done = terminated or truncated
            ep_ret += float(r)
        returns.append(ep_ret)
        successes.append(float(inf.get("is_success", False)))
    eval_env.close()
    return float(np.mean(returns)), float(np.mean(successes))


In [ ]:
# Training loop
obs, info = env.reset(seed=CFG.seed)
start_t = time.time()
episode_return = 0.0
episode_count = 0
last_log = {}

for t in range(1, CFG.total_steps + 1):
    if t < CFG.warmup_steps:
        action = env.action_space.sample()
    else:
        action = agent.act(obs, deterministic=False)

    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated

    # Replay buffer is GPU-native; pass tensors directly on target device.
    replay.add(
        torch.as_tensor(obs, dtype=torch.float32, device=device),
        torch.as_tensor(action, dtype=torch.float32, device=device),
        float(reward),
        torch.as_tensor(next_obs, dtype=torch.float32, device=device),
        float(done),
    )

    obs = next_obs
    episode_return += float(reward)

    if done:
        episode_count += 1
        obs, info = env.reset()
        episode_return = 0.0

    if replay.size >= CFG.batch_size and t >= CFG.warmup_steps:
        for _ in range(CFG.updates_per_step):
            last_log = agent.update(replay, CFG.batch_size)

    if t % CFG.log_every == 0:
        elapsed = time.time() - start_t
        sps = t / max(elapsed, 1e-6)
        print(
            f"step={t:,} ep={episode_count} replay={replay.size:,} "
            f"critic={last_log.get('critic_loss', float('nan')):.3f} "
            f"actor={last_log.get('actor_loss', float('nan')):.3f} "
            f"alpha={last_log.get('alpha', float('nan')):.3f} sps={sps:.1f}"
        )

    if t % CFG.eval_every == 0:
        mean_ret, success = run_eval(agent, CFG.eval_episodes)
        print(f"[eval] step={t:,} mean_return={mean_ret:.2f} success={success:.2f}")

env.close()
print("Training complete")


In [ ]:
# Save / load example
ckpt_path = "gsac_s1m1_sample.pt"
agent.save(ckpt_path)
print("saved:", ckpt_path)

# To reload later:
# agent.load(ckpt_path)
